# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [ ]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from bs4 import BeautifulSoup
import requests
from urllib.parse import urljoin
from openai import OpenAI
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [ ]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')
gemini_api_key = os.getenv('GEMINI_API_KEY')
gemmini_model = os.getenv('GEMINI_MODEL')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL_GPT = 'gpt-5-nano'
MODEL_LLAMA = "llama3.2"

openai = OpenAI()

OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
gemini = OpenAI(base_url="https://gemini.api.openai.com/v1", api_key=gemini_api_key)

In [ ]:
# A class to represent a Webpage

# Some websites need you to use proper headers when fetching them:

class Website:
    def __init__(self, url):
        self.url = url
        self.title = "No title found"
        self.text = ""
        self.soup = None  # Inicjalizujemy jako None
        
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
        }

        try:
            session = requests.Session()
            # Pierwsze zapytanie, by złapać ciasteczka
            # session.get(url, headers=headers, timeout=10, verify=False)
            response = session.get(url, headers=headers, allow_redirects=True, timeout=10, verify=False)
            # Sprawdzamy czy pobieranie się udało (kod 200)
            response.raise_for_status() 
            
            self.body = response.content
            self.soup = BeautifulSoup(self.body, 'html.parser')
            
            if self.soup.title:
                self.title = self.soup.title.string
            
            if self.soup.body:
                self.text = self.soup.body.get_text(separator="\n", strip=True)
        except requests.exceptions.SSLError:
            print(f"SSL Error for {url}. Try updating your certificates or use verify=False.")        
        except Exception as e:
            print(f"Website error for {url}: {e}")

    # def get_links(self):
    #     # Kluczowe: sprawdź czy soup w ogóle istnieje
    #     if not self.soup:
    #         return []
            
    #     links = []
    #     for a in self.soup.find_all('a', href=True):
    #         text = a.get_text(strip=True)
    #         href = a['href']
    #         # Filtrujemy tylko linki z tekstem lub te 'podejrzane' o repertuar
    #         if text or "repertuar" in href.lower():
    #             # Używamy self.url jako bazy
    #             full_url = urljoin(self.url, href)
    #             links.append({"text": text, "url": full_url})
        
    #     return links
    def get_links(self):
        if not self.soup:
            return []
        
        seen_urls = set()
        candidates = []
        keywords = ["repertuar", "program", "bilety", "schedule"]
        
        for a in self.soup.find_all('a', href=True):
            href = urljoin(self.url, a['href'])
            
            # Unikamy duplikatów i linków typu "pusty hashtag"
            if href in seen_urls or href.strip() == self.url or href.endswith("#"):
                continue
                
            text = a.get_text(strip=True).lower()
            href_lower = href.lower()
            
            if any(kw in text or kw in href_lower for kw in keywords):
                seen_urls.add(href)
                candidates.append({"text": a.get_text(strip=True), "url": href})
                
        return candidates
    
    def get_raw(self):
        try:
            for element in self.soup(["script", "style", "head", "footer", "noscript", "svg", "form"]):
                element.decompose()
            allowed_attrs = ["href", "title", "class"]
            for tag in self.soup.find_all(True):
                tag.attrs = {name: value for name, value in tag.attrs.items() if name in allowed_attrs}
            return self.soup.prettify()
        except:
            return b""  # return empty bytes on error 
    def get_content(self):
        try:
            return f"Webpage Title:\n{self.title}\nWebpage Contents:\n{self.text}\n\n"
        except:
            return "" 

In [ ]:
url = "https://kinoluna.waw.pl/"
website = Website(url)
links = website.get_links()
print("\n".join([link['url'] for link in links]))


## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [ ]:
# link_system_prompt = """
# You are a helpfull asistant.
# Search the navigation section (<nav> or <ul>) in the provided HTML code. 
# Find the link whose text is 'Repertuar' or whose href attribute contains the word 'repertuar'. 
# Link this link to the main domain. Return the result in JSON format only.

# Return JSON in this format:
# {
#   "repertuar_link": "https://full.url/repertuar"
# }

# """

# link_system_prompt = """
# You are a deterministic URL selector.
# Your goal is to find the link to the cinema's movie schedule (repertuar) from a provided list of links.

# Rules:
# 1. Analyze both the link text and the URL path. 
# 2. Keywords to prioritize: "repertuar", "schedule", "program", "calendar".
# 3. If the selected link is relative (e.g., "/repertuar"), you MUST prepend the base URL.
# 4. If the link is already absolute (starts with http), return it as is.
# 5. Respond ONLY with a valid JSON object.

# Format:
# {
#   "repertuar_link": "https://full.url/path"
# }
# """

link_system_prompt = """
You are a precision extraction tool. You convert link lists into a specific JSON format.

RULE: Your output must ALWAYS be a single JSON object with the key "repertuar_link".
RULE: Do not use keys like "key", "value", or "text".

EXAMPLE:
Input: [{"text": "Repertuar", "url": "https://cinema.com/schedule"}]
Output: {"repertuar_link": "https://cinema.com/schedule"}

NOW PROCESS THE INPUT BELOW:
"""

In [ ]:
# def get_link_user_prompt(url):
#     user_prompt = f"""
# You are given a raw content from the cinema website {url}.

# Your task:
# - Identify the link that leads to the repertuar (movie schedule) page.
# - Return ONLY the single most relevant link.
# - If the link is relative (e.g. "/repertuar"), return full url.
# - Respond in JSON format.

# Example response:
# {{"repertuar_link": "/repertuar"}}

# Content:
# """
#     website = Website(url)
#     content = website.get_raw()
#     # candidate_links = filter_same_domain(url, links)
#     user_prompt += content
#     return user_prompt
def get_link_user_prompt(url):
    website = Website(url)
    links = website.get_links()
    links_formatted = json.dumps(links, indent=2, ensure_ascii=False)
    user_prompt = f"""
Base URL: {url}
Link Candidates:
{links_formatted}
Identify the repertuar link from the list above.
"""
    return user_prompt


In [ ]:
print(get_link_user_prompt("https://kinomuranow.pl/"))

In [ ]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')
gemini_api_key = os.getenv('GEMINI_API_KEY')
gemmini_model = os.getenv('GEMINI_MODEL')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
# MODEL = 'gpt-5-nano'
openai = OpenAI()

OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
gemini = OpenAI(base_url="https://gemini.api.openai.com/v1", api_key=gemini_api_key)
MODEL = "llama3.2"

In [ ]:
print(get_link_user_prompt("https://kinoluna.waw.pl/"))

In [ ]:
def extract_repertuar_link(ai_response):
    if not isinstance(ai_response, dict):
        return None
        
    # 1. Jeśli model był grzeczny i użył właściwego klucza
    if "repertuar_link" in ai_response:
        return ai_response["repertuar_link"]
    
    # 2. Jeśli model użył struktury key-value (jak w Twoim przypadku)
    if ai_response.get("key") and ai_response.get("value"):
        # Sprawdzamy, czy 'value' to URL, jeśli tak - to jest to!
        if str(ai_response["value"]).startswith("http"):
            return ai_response["value"]

    # 3. Ostateczność: Przeszukaj wszystkie wartości i weź pierwszą, która jest linkiem
    for val in ai_response.values():
        if isinstance(val, str) and val.startswith("http"):
            return val
            
    return None

In [ ]:
def select_relevant_link(url):
    print(f"Selecting relevant Repertuar link for {url} by calling {MODEL_LLAMA}")
    response = ollama.chat.completions.create(
        model=MODEL_LLAMA,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_link_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    # print(f"Response: {response.choices[0].message.content}")
    result = response.choices[0].message.content
    link = json.loads(result)
    # print(f"Link: {link}", "\n\n")
    # if links['links'] is not None:
    #     print(f"Found {len(links['links'])} relevant links")
    # else:
    #     print("No relevant links found")
    # return link
    return extract_repertuar_link(link)


In [ ]:
print(select_relevant_link("https://kinoteka.pl/"))


In [ ]:
print(select_relevant_link("https://kinoluna.waw.pl/"))

In [ ]:
# print(get_links_user_prompt("https://kinomuranow.pl/"))
print(select_relevant_link("https://kinomuranow.pl/"))

In [ ]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')
gemini_api_key = os.getenv('GEMINI_API_KEY')
gemmini_model = os.getenv('GEMINI_MODEL')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
# MODEL = 'gpt-5-nano'
openai = OpenAI()

OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
gemini = OpenAI(base_url="https://gemini.api.openai.com/v1", api_key=gemini_api_key)
MODEL = "llama3.2"

link_system_prompt = """
You are provided with a list of links to cinemas webpage.
You are able to find the most relevant link to "Repertuar" page about the movies being shown in the cinema,
such as links to a Repertuar page, or a Movies page.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "repertuar", "url": "https://full.url/goes/here/repertuar"},
    ]
}
"""

def get_links_user_prompt(cinemas_website_url):
    user_prompt = f"""
Here is the list of links to the cinemas website -
Please decide which of these are relevant web links for repertuar page about the movies being shown in the cinema, 
respond with the full https URL in JSON format.
Include only relevant link.

Links (some might be relative links):

"""
    for url in cinemas_website_url:
        links += fetch_website_links(url)

    user_prompt += "\n".join(links)
    return user_prompt


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>